In [1]:
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import f
import ruptures as rpt
from statsmodels.stats.diagnostic import breaks_cusumolsresid
from scipy.stats import ttest_ind
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

sns.set_style("whitegrid")

In [ ]:
# Read the CSV with no header because the file contains metadata/title rows before the actual table header.
file_path = "HK_CPI_data.csv"
raw = pd.read_csv(file_path, header=None)

In [3]:
#--------------------------- cleaning the file ----------------------------

# Locate the true header row by finding where col0 == "Year" and col1 == "Month".
# This is more robust than hard-coding a row number because source files may shift.
hdr_idx = raw.index[
    (raw.iloc[:, 0].astype(str).str.strip().eq("Year"))
    & (raw.iloc[:, 1].astype(str).str.strip().eq("Month"))
][0]

# The table uses a 2-row grouped header above the "Year/Month" row:
# - h1: high-level group (e.g., CPI type)
# - h2: metric (e.g., Index, YoY change, MoM change)
h1 = raw.iloc[hdr_idx - 2].ffill()  # forward-fill group names across merged-like cells
h2 = raw.iloc[hdr_idx - 1]

# Build clean, unique column names by combining h1 and h2.
new_cols = []
seen = {}  # tracks duplicates to enforce uniqueness

for i in range(raw.shape[1]):
    if i == 0:
        col = "Year"
    elif i == 1:
        col = "Month"
    else:
        a = str(h1[i]).strip() if pd.notna(h1[i]) else ""
        b = str(h2[i]).strip() if pd.notna(h2[i]) else ""

        # Prefer "Group - Metric"; fall back to whichever exists; else generated name.
        col = f"{a} - {b}" if a and b else (a or b or f"col_{i}")

        # Normalize symbols/text to avoid awkward column names in downstream code.
        col = col.replace("%", "pct")

        # Remove non-breaking spaces and collapse extra whitespace to prevent hidden
        # near-duplicates (e.g., visually same but technically different names).
        col = col.replace("\xa0", " ")
        col = " ".join(col.split())

    # Ensure uniqueness in case normalization produces duplicate names.
    if col in seen:
        seen[col] += 1
        col = f"{col}_{seen[col]}"
    else:
        seen[col] = 0

    new_cols.append(col)

# Extract data rows only (everything below the header marker row) and assign new columns.
cpi_df = raw.iloc[hdr_idx + 1 :].copy()
cpi_df.columns = new_cols

# Clean and convert all value columns (exclude Year/Month at positions 0 and 1).
for col in cpi_df.columns[2:]:
    cpi_df[col] = (
        cpi_df[col]
        .astype(str)
        # Remove bracketed footnotes like "[1]" that break numeric parsing.
        .str.replace(r"\s*\[.*?\]", "", regex=True)
        # Standardize missing-value markers before numeric conversion.
        .replace({"N.A.": pd.NA, "nan": pd.NA, "": pd.NA})
    )
    # Coerce invalid strings to NaN so numeric operations work safely.
    cpi_df[col] = pd.to_numeric(cpi_df[col], errors="coerce")

# Convert Year/Month to year and month numbers only (nullable integers).
year_dt = pd.to_datetime(
    cpi_df["Year"].astype("string").str.strip(),
    format="%Y",
    errors="coerce",
)

month_str = cpi_df["Month"].astype("string").str.strip()
month_num = pd.to_numeric(month_str, errors="coerce")
month_num = month_num.fillna(pd.to_datetime(month_str, format="%b", errors="coerce").dt.month)
month_num = month_num.fillna(pd.to_datetime(month_str, format="%B", errors="coerce").dt.month)

cpi_df["Year"] = year_dt.dt.year.astype("Int64")
cpi_df["Month"] = month_num.astype("Int64")

# Keep only rows with a valid year (drops footer/notes rows).
cpi_df = cpi_df[cpi_df["Year"].notna()].reset_index(drop=True)

# Show only Year and Month preview.
cpi_df[["Year", "Month"]].head()

# Quick sanity check of the cleaned output.
cpi_df.head()

,Year,Month,Composite Consumer Price Index - Index,Composite Consumer Price Index - Year-on-year pct change,Composite Consumer Price Index - Month-to-month pct change,Consumer Price Index (A) - Index,Consumer Price Index (A) - Year-on-year pct change,Consumer Price Index (A) - Month-to-month pct change,Consumer Price Index (B) - Index,Consumer Price Index (B) - Year-on-year pct change,Consumer Price Index (B) - Month-to-month pct change,Consumer Price Index (C) - Index,Consumer Price Index (C) - Year-on-year pct change,Consumer Price Index (C) - Month-to-month pct change
0,1975,<NA>,NaN,NaN,NaN,12.7,NaN,NaN,13.2,NaN,NaN,11.9,NaN,NaN
1,1976,<NA>,NaN,NaN,NaN,13.1,3.4,NaN,13.7,4.0,NaN,12.5,4.4,NaN
2,1977,<NA>,NaN,NaN,NaN,13.9,5.8,NaN,14.4,5.5,NaN,13.1,5.0,NaN
3,1978,<NA>,NaN,NaN,NaN,14.7,5.9,NaN,15.3,5.9,NaN,13.8,5.8,NaN
4,1979,<NA>,NaN,NaN,NaN,16.4,11.6,NaN,17.0,11.5,NaN,15.5,12.4,NaN


In [15]:
cpi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 14 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Year                                                        669 non-null    Int64  
 1   Month                                                       618 non-null    Int64  
 2   Composite Consumer Price Index - Index                      588 non-null    float64
 3   Composite Consumer Price Index - Year-on-year pct change    575 non-null    float64
 4   Composite Consumer Price Index - Month-to-month pct change  542 non-null    float64
 5   Consumer Price Index (A) - Index                            669 non-null    float64
 6   Consumer Price Index (A) - Year-on-year pct change          656 non-null    float64
 7   Consumer Price Index (A) - Month-to-month pct change        617 non-null    float64
 8   

In [4]:
# Shorten cpi_df column names
short_name_map = {
    "Consumer Price Index": "CPI",
    "Composite CPI": "CCPI",
    "CPI (A)": "CPI_A",
    "CPI (B)": "CPI_B",
    "CPI (C)": "CPI_C",
    "Year-on-year pct change": "yoy_pct",
    "Month-to-month pct change": "mom_pct",
    "Index": "idx",
}

cpi_df = cpi_df.rename(
    columns=lambda col: col if col in ["Year", "Month"] else (
        col.replace("Consumer Price Index", short_name_map["Consumer Price Index"])
           .replace("Composite CPI", short_name_map["Composite CPI"])
           .replace("CPI (A)", short_name_map["CPI (A)"])
           .replace("CPI (B)", short_name_map["CPI (B)"])
           .replace("CPI (C)", short_name_map["CPI (C)"])
           .replace("Year-on-year pct change", short_name_map["Year-on-year pct change"])
           .replace("Month-to-month pct change", short_name_map["Month-to-month pct change"])
           .replace("Index", short_name_map["Index"])
           .replace(" - ", "_")
           .replace(" ", "")
    )
)

cpi_df.columns

Index(['Year', 'Month', 'CCPI_idx', 'CCPI_yoy_pct', 'CCPI_mom_pct',
       'CPI_A_idx', 'CPI_A_yoy_pct', 'CPI_A_mom_pct', 'CPI_B_idx',
       'CPI_B_yoy_pct', 'CPI_B_mom_pct', 'CPI_C_idx', 'CPI_C_yoy_pct',
       'CPI_C_mom_pct'],
      dtype='object')

In [17]:
cpi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Year           669 non-null    Int64  
 1   Month          618 non-null    Int64  
 2   CCPI_idx       588 non-null    float64
 3   CCPI_yoy_pct   575 non-null    float64
 4   CCPI_mom_pct   542 non-null    float64
 5   CPI_A_idx      669 non-null    float64
 6   CPI_A_yoy_pct  656 non-null    float64
 7   CPI_A_mom_pct  617 non-null    float64
 8   CPI_B_idx      669 non-null    float64
 9   CPI_B_yoy_pct  656 non-null    float64
 10  CPI_B_mom_pct  617 non-null    float64
 11  CPI_C_idx      669 non-null    float64
 12  CPI_C_yoy_pct  656 non-null    float64
 13  CPI_C_mom_pct  617 non-null    float64
dtypes: Int64(2), float64(12)
memory usage: 74.6 KB


In [5]:
#Filter for only year>2000
cpi_df = cpi_df[cpi_df["Year"] >= 2000].reset_index(drop=True)

In [19]:
cpi_df.head()

,Year,Month,CCPI_idx,CCPI_yoy_pct,CCPI_mom_pct,CPI_A_idx,CPI_A_yoy_pct,CPI_A_mom_pct,CPI_B_idx,CPI_B_yoy_pct,CPI_B_mom_pct,CPI_C_idx,CPI_C_yoy_pct,CPI_C_mom_pct
0,2000,<NA>,70.9,-3.8,NaN,69.3,-3.0,NaN,71.1,-3.9,NaN,72.4,-4.5,NaN
1,2001,<NA>,69.8,-1.6,NaN,68.2,-1.7,NaN,69.9,-1.6,NaN,71.3,-1.5,NaN
2,2002,<NA>,67.6,-3.0,NaN,66.0,-3.2,NaN,67.8,-3.1,NaN,69.3,-2.8,NaN
3,2003,<NA>,65.9,-2.6,NaN,64.6,-2.1,NaN,65.9,-2.7,NaN,67.3,-2.9,NaN
4,2004,<NA>,65.6,-0.4,NaN,64.6,0.0,NaN,65.6,-0.5,NaN,66.8,-0.9,NaN


In [6]:
# Use monthly records only (annual rows have Month = <NA>)
monthly = cpi_df[cpi_df["Month"].notna()].copy()
monthly["Date"] = pd.to_datetime(
    dict(year=monthly["Year"].astype(int), month=monthly["Month"].astype(int), day=1)
)
monthly = monthly.sort_values("Date").reset_index(drop=True)

## Normal EDA

In [21]:
# 4) Interactive correlation heatmap across numeric variables
corr = monthly.select_dtypes(include="number").corr()
fig = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Interactive Correlation Heatmap (Monthly CPI Variables)",
)
fig.update_layout(template="plotly_white", dragmode="pan")
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

In [22]:
# 5) Interactive distribution comparison of inflation measures
dist_df = monthly[["CCPI_yoy_pct", "CPI_A_yoy_pct", "CPI_B_yoy_pct", "CPI_C_yoy_pct"]].copy()
dist_long = dist_df.melt(var_name="Series", value_name="YoY_pct")

fig = px.box(
    dist_long,
    x="Series",
    y="YoY_pct",
    points="outliers",
    title="Interactive Distribution of YoY Inflation Rates",
)
fig.update_layout(template="plotly_white", yaxis_title="YoY %", dragmode="pan")
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})


In [23]:
# 7) Additional EDA: seasonality heatmap (average YoY by year-month)
seasonal = monthly[["Date", "CCPI_yoy_pct"]].dropna().copy()
seasonal["Year"] = seasonal["Date"].dt.year
seasonal["MonthNum"] = seasonal["Date"].dt.month
pivot = seasonal.pivot_table(index="Year", columns="MonthNum", values="CCPI_yoy_pct", aggfunc="mean")

fig = px.imshow(
    pivot,
    labels={"x": "Month", "y": "Year", "color": "Avg YoY %"},
    color_continuous_scale="RdYlBu_r",
    aspect="auto",
    title="Seasonality Heatmap: Average CCPI YoY Inflation",
)
fig.update_layout(template="plotly_white", dragmode="pan")
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

## Investigating relationship with major global/regional events

In [24]:
# unmerged_events 

# event_periods = pd.DataFrame(
#     [
#         # Your original events (kept)
#         ("Dot-com bust", "2000-03-01", "2002-12-01", "Global demand slowdown"),
#         ("SARS outbreak", "2003-03-01", "2003-07-01", "Tourism/services disruption in Hong Kong"),
#         ("Global Financial Crisis", "2008-09-01", "2009-12-01", "Global recession and demand shock"),
#         ("Eurozone debt crisis", "2010-05-01", "2012-12-01", "External demand and risk sentiment shock"),
#         ("Oil price collapse", "2014-07-01", "2016-02-01", "Energy and transport cost disinflation"),
#         ("US-China trade tensions", "2018-03-01", "2019-12-01", "Trade, supply chain and import-price uncertainty"),
#         ("Social unrest in Hong Kong", "2019-06-01", "2020-06-01", "Local demand and tourism weakness"),
#         ("COVID-19 pandemic", "2020-01-01", "2023-05-01", "Global + local demand/supply shock"),
#         ("Global energy shock (post-invasion)", "2022-02-01", "2023-12-01", "Energy and commodity inflation pressure"),
#         ("Red Sea shipping disruptions", "2023-10-01", "2025-12-01", "Freight cost and delivery-time pressures"),
#         ("China property sector stress", "2021-09-01", "2025-12-01", "Mainland/HK demand and sentiment spillovers"),
#         ("Global banking stress (SVB/Credit Suisse)", "2023-03-01", "2023-06-01", "Tighter financial conditions"),
#         # Added events (new)
#         ("Taper tantrum (Fed communications shock)", "2013-05-01", "2013-07-01", "Global risk-off, EM capital outflows; pressure on yields/borrowing costs"),
#         ("China stock market turbulence & RMB moves", "2015-06-01", "2016-02-01", "Asset volatility and currency moves; confidence & trade/FX uncertainty"),
#         ("RMB devaluation episode (Aug 2015)", "2015-08-11", "2015-09-01", "One-off currency pass-through risk to import prices / sentiment"),
#         ("Brexit vote (global risk shock)", "2016-06-23", "2016-07-15", "Global risk sentiment & safe-haven flows"),
#         ("Japan earthquake & Fukushima shock (supply/energy)", "2011-03-11", "2011-06-01", "Energy/supply-chain disruptions, commodity price impacts"),
#         ("Global inflation surge & supply-chain shock (post-COVID)", "2021-04-01", "2023-12-01", "Import price rises, shipping costs, wages → upward CPI pressure"),
#         ("Fed rate-hike cycle & HK pass-through", "2022-03-01", "2024-12-01", "Higher US rates → HK mortgage/borrowing costs, property, financial conditions (HKMA pass-through)"),
#         ("China reopening (end of Zero-COVID)", "2022-12-01", "2023-06-01", "Tourism & goods demand rebound; commodity and travel-price effects"),
#         # Smaller/overlapping events you already had but consider merging
#         ("Asian Financial Crisis after-effects (suggest merge)", "2000-01-01", "2003-12-01", "HK deflation / housing effects — consider merging with 'HK property market slump'"),
#         ("HK property market slump", "2000-01-01", "2004-12-01", "Housing/rent collapse causing prolonged deflation"),
#     ],
#     columns=["Event", "Start", "End", "Channel"],
# )

# event_periods["Start"] = pd.to_datetime(event_periods["Start"])
# event_periods["End"] = pd.to_datetime(event_periods["End"])
# print(event_periods)

In [7]:

event_periods = pd.DataFrame(
    [

        # Early structural deflation episode (MERGED)
        (
            "Early-2000s HK deflation & property slump",
            "2000-01-01",
            "2004-12-01",
            "Housing/rent collapse + weak demand produced sustained CPI deflation"
        ),

        # Independent macro shocks
        (
            "SARS outbreak",
            "2003-03-01",
            "2003-07-01",
            "Tourism and local demand collapse"
        ),

        (
            "Global Financial Crisis",
            "2008-09-01",
            "2009-12-01",
            "Global recession and demand shock"
        ),

        (
            "US subprime crisis onset",
            "2007-08-01",
            "2008-03-01",
            "Pre-GFC financial tightening and confidence shock"
        ),

        (
            "Eurozone debt crisis",
            "2010-05-01",
            "2012-12-01",
            "External demand and financial uncertainty"
        ),

        (
            "Statutory Minimum Wage introduction",
            "2011-05-01",
            "2012-12-01",
            "Labor-cost push inflation in services"
        ),

        (
            "Japan earthquake & Fukushima supply shock",
            "2011-03-11",
            "2011-06-01",
            "Energy and supply-chain disruptions"
        ),

        # China-driven disinflation episode (MERGED)
        (
            "China slowdown & RMB adjustment episode",
            "2014-07-01",
            "2016-02-01",
            "China growth slowdown + FX moves + commodity disinflation"
        ),

        (
            "Taper tantrum",
            "2013-05-01",
            "2013-07-01",
            "Global rate shock and financial tightening"
        ),

        (
            "Brexit risk shock",
            "2016-06-23",
            "2016-07-15",
            "Global risk sentiment volatility"
        ),

        (
            "US-China trade tensions",
            "2018-03-01",
            "2019-12-01",
            "Trade and import-price uncertainty"
        ),

        (
            "Social unrest in Hong Kong",
            "2019-06-01",
            "2020-06-01",
            "Tourism and domestic demand collapse"
        ),

        (
            "COVID-19 pandemic (core demand shock)",
            "2020-01-01",
            "2021-03-01",
            "Global and local demand collapse"
        ),

        # Supply-chain inflation episode (MERGED)
        (
            "Post-COVID supply-chain inflation",
            "2021-04-01",
            "2023-12-01",
            "Shipping costs, import prices and goods inflation"
        ),

        (
            "China property sector stress",
            "2021-09-01",
            "2025-12-01",
            "Mainland demand and sentiment spillovers"
        ),

        (
            "Global energy shock (Ukraine war)",
            "2022-02-01",
            "2023-12-01",
            "Energy and commodity inflation"
        ),

        (
            "Fed tightening cycle & HK pass-through",
            "2022-03-01",
            "2024-12-01",
            "Interest rate transmission to housing and financial conditions"
        ),

        (
            "China reopening rebound",
            "2022-12-01",
            "2023-06-01",
            "Tourism and demand recovery"
        ),

        (
            "Global banking stress",
            "2023-03-01",
            "2023-06-01",
            "Financial tightening shock"
        ),

        (
            "Red Sea shipping disruptions",
            "2023-10-01",
            "2025-12-01",
            "Freight cost inflation pressures"
        ),

    ],
    columns=["Event", "Start", "End", "Channel"],
)

event_periods["Start"] = pd.to_datetime(event_periods["Start"])
event_periods["End"] = pd.to_datetime(event_periods["End"])

print(event_periods)

                                        Event      Start        End  \
0   Early-2000s HK deflation & property slump 2000-01-01 2004-12-01   
1                               SARS outbreak 2003-03-01 2003-07-01   
2                     Global Financial Crisis 2008-09-01 2009-12-01   
3                    US subprime crisis onset 2007-08-01 2008-03-01   
4                        Eurozone debt crisis 2010-05-01 2012-12-01   
5         Statutory Minimum Wage introduction 2011-05-01 2012-12-01   
6   Japan earthquake & Fukushima supply shock 2011-03-11 2011-06-01   
7     China slowdown & RMB adjustment episode 2014-07-01 2016-02-01   
8                               Taper tantrum 2013-05-01 2013-07-01   
9                           Brexit risk shock 2016-06-23 2016-07-15   
10                    US-China trade tensions 2018-03-01 2019-12-01   
11                 Social unrest in Hong Kong 2019-06-01 2020-06-01   
12      COVID-19 pandemic (core demand shock) 2020-01-01 2021-03-01   
13    

In [8]:

def label_events(date_value):
    active = event_periods.loc[
        (event_periods["Start"] <= date_value) & (date_value <= event_periods["End"]),
        "Event",
    ].tolist()
    return " | ".join(active) if active else "Non-event period"

monthly["Event_Period"] = monthly["Date"].apply(label_events)
monthly["In_Event_Window"] = monthly["Event_Period"].ne("Non-event period")
monthly["Event_Count"] = monthly["Date"].apply(
    lambda d: int(((event_periods["Start"] <= d) & (d <= event_periods["End"])).sum())
)

event_periods

,Event,Start,End,Channel
0,Early-2000s HK deflation & property slump,2000-01-01,2004-12-01,Housing/rent collapse + weak demand produced s...
1,SARS outbreak,2003-03-01,2003-07-01,Tourism and local demand collapse
2,Global Financial Crisis,2008-09-01,2009-12-01,Global recession and demand shock
3,US subprime crisis onset,2007-08-01,2008-03-01,Pre-GFC financial tightening and confidence shock
4,Eurozone debt crisis,2010-05-01,2012-12-01,External demand and financial uncertainty
5,Statutory Minimum Wage introduction,2011-05-01,2012-12-01,Labor-cost push inflation in services
6,Japan earthquake & Fukushima supply shock,2011-03-11,2011-06-01,Energy and supply-chain disruptions
7,China slowdown & RMB adjustment episode,2014-07-01,2016-02-01,China growth slowdown + FX moves + commodity d...
8,Taper tantrum,2013-05-01,2013-07-01,Global rate shock and financial tightening
9,Brexit risk shock,2016-06-23,2016-07-15,Global risk sentiment volatility


In [27]:
monthly

,Year,Month,CCPI_idx,CCPI_yoy_pct,CCPI_mom_pct,CPI_A_idx,CPI_A_yoy_pct,CPI_A_mom_pct,CPI_B_idx,CPI_B_yoy_pct,CPI_B_mom_pct,CPI_C_idx,CPI_C_yoy_pct,CPI_C_mom_pct,Date,Event_Period,In_Event_Window,Event_Count
0,2000,1,71.5,-5.3,-0.7,69.9,-4.2,-0.5,71.8,-5.7,-0.6,72.9,-6.2,-1.0,2000-01-01,Early-2000s HK deflation & property slump,True,1
1,2000,2,71.4,-5.1,-0.1,69.9,-4.2,0.0,71.7,-5.4,-0.1,72.9,-5.8,0.0,2000-02-01,Early-2000s HK deflation & property slump,True,1
2,2000,3,71.2,-5.0,-0.4,69.5,-4.0,-0.5,71.4,-5.3,-0.4,72.7,-5.7,-0.3,2000-03-01,Early-2000s HK deflation & property slump,True,1
3,2000,4,71.2,-4.4,0.1,69.5,-3.4,0.0,71.4,-4.6,0.0,72.9,-5.2,0.3,2000-04-01,Early-2000s HK deflation & property slump,True,1
4,2000,5,71.1,-4.5,-0.2,69.4,-3.4,-0.2,71.2,-4.6,-0.3,72.8,-5.6,-0.2,2000-05-01,Early-2000s HK deflation & property slump,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,2025,8,109.0,1.1,0.1,111.8,1.5,0.1,107.7,1.0,0.1,107.5,0.9,0.2,2025-08-01,China property sector stress | Red Sea shippin...,True,2
308,2025,9,109.1,1.1,0.1,111.9,1.5,0.1,107.8,0.9,0.1,107.5,0.8,0.0,2025-09-01,China property sector stress | Red Sea shippin...,True,2
309,2025,10,109.4,1.2,0.3,112.1,1.4,0.1,108.1,1.1,0.3,108.0,1.0,0.4,2025-10-01,China property sector stress | Red Sea shippin...,True,2
310,2025,11,109.4,1.2,0.0,112.1,1.5,0.0,108.1,1.1,0.0,108.0,1.1,0.0,2025-11-01,China property sector stress | Red Sea shippin...,True,2


### Graphs

In [28]:
# 1) Interactive CPI index trends with event windows
fig = go.Figure()
for col in ["CCPI_idx", "CPI_A_idx", "CPI_B_idx", "CPI_C_idx"]:
    fig.add_trace(
        go.Scatter(
            x=monthly["Date"],
            y=monthly[col],
            mode="lines",
            name=col,
            hovertemplate="Date=%{x|%Y-%m}<br>Value=%{y:.2f}<extra>" + col + "</extra>",
        )
    )

for _, row in event_periods.iterrows():
    fig.add_vrect(
        x0=row["Start"],
        x1=row["End"],
        fillcolor="LightSalmon",
        opacity=0.12,
        layer="below",
        line_width=0,
        annotation_text=row["Event"],
        annotation_position="top left",
    )

fig.update_layout(
    title="Interactive CPI Index Trends (with Major Event Periods)",
    xaxis_title="Date",
    yaxis_title="Index",
    hovermode="x unified",
    legend_title_text="Series",
    template="plotly_white",
    dragmode="pan",
)
fig.update_xaxes(rangeslider_visible=True)
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

In [29]:
# 2) Interactive year-on-year inflation rates with event windows
fig = go.Figure()
for col in ["CCPI_yoy_pct", "CPI_A_yoy_pct", "CPI_B_yoy_pct", "CPI_C_yoy_pct"]:
    fig.add_trace(
        go.Scatter(
            x=monthly["Date"],
            y=monthly[col],
            mode="lines",
            name=col,
            hovertemplate="Date=%{x|%Y-%m}<br>YoY=%{y:.2f}%<extra>" + col + "</extra>",
        )
    )

fig.add_hline(y=0, line_dash="dash", line_color="black")
for _, row in event_periods.iterrows():
    fig.add_vrect(
        x0=row["Start"],
        x1=row["End"],
        fillcolor="LightGreen",
        opacity=0.10,
        layer="below",
        line_width=0,
    )

fig.update_layout(
    title="Interactive Year-on-Year CPI Change (with Major Event Periods)",
    xaxis_title="Date",
    yaxis_title="YoY %",
    hovermode="x unified",
    legend_title_text="Series",
    template="plotly_white",
    dragmode="pan",
)
fig.update_xaxes(rangeslider_visible=True)
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

In [30]:
# 3) Interactive month-to-month changes (volatility focus)
fig = go.Figure()
for col in ["CCPI_mom_pct", "CPI_A_mom_pct", "CPI_B_mom_pct", "CPI_C_mom_pct"]:
    fig.add_trace(
        go.Scatter(
            x=monthly["Date"],
            y=monthly[col],
            mode="lines",
            name=col,
            hovertemplate="Date=%{x|%Y-%m}<br>MoM=%{y:.2f}%<extra>" + col + "</extra>",
        )
    )

fig.add_hline(y=0, line_dash="dash", line_color="black")
for _, row in event_periods.iterrows():
    fig.add_vrect(
        x0=row["Start"],
        x1=row["End"],
        fillcolor="LightBlue",
        opacity=0.08,
        layer="below",
        line_width=0,
    )

fig.update_layout(
    title="Interactive Month-to-Month CPI Change (with Major Event Periods)",
    xaxis_title="Date",
    yaxis_title="MoM %",
    hovermode="x unified",
    legend_title_text="Series",
    template="plotly_white",
    dragmode="pan",
)
fig.update_xaxes(rangeslider_visible=True)
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

In [31]:
# 6) Additional EDA: 12-month rolling YoY trend
rolling_df = monthly[["Date", "CCPI_yoy_pct", "CPI_A_yoy_pct", "CPI_B_yoy_pct", "CPI_C_yoy_pct"]].copy()
for col in ["CCPI_yoy_pct", "CPI_A_yoy_pct", "CPI_B_yoy_pct", "CPI_C_yoy_pct"]:
    rolling_df[col + "_roll12"] = rolling_df[col].rolling(12, min_periods=6).mean()

fig = go.Figure()
for col, name in [
    ("CCPI_yoy_pct_roll12", "CCPI (12M avg)"),
    ("CPI_A_yoy_pct_roll12", "A (12M avg)"),
    ("CPI_B_yoy_pct_roll12", "B (12M avg)"),
    ("CPI_C_yoy_pct_roll12", "C (12M avg)"),
]:
    fig.add_trace(
        go.Scatter(
            x=rolling_df["Date"],
            y=rolling_df[col],
            mode="lines",
            name=name,
        )
    )

for _, row in event_periods.iterrows():
    fig.add_vrect(
        x0=row["Start"],
        x1=row["End"],
        fillcolor="LightSalmon",
        opacity=0.08,
        layer="below",
        line_width=0,
    )

fig.update_layout(
    title="12-Month Rolling Average of YoY CPI Inflation",
    xaxis_title="Date",
    yaxis_title="YoY % (12M avg)",
    hovermode="x unified",
    template="plotly_white",
    dragmode="pan",
)
fig.update_xaxes(rangeslider_visible=True)
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

In [32]:
# 8) Additional EDA: event vs non-event inflation regime comparison
regime_df = monthly.copy()

if "Event_Count" not in regime_df.columns:
    if "Event_Period" in regime_df.columns:
        regime_df["Event_Count"] = (
            regime_df["Event_Period"]
            .fillna("None")
            .replace("None", "")
            .apply(lambda x: 0 if x == "" else len(str(x).split(" | ")))
        )
    else:
        regime_df["Event_Count"] = regime_df.get("In_Event_Window", False).astype(int)

if "In_Event_Window" not in regime_df.columns:
    regime_df["In_Event_Window"] = regime_df["Event_Count"] > 0

regime_df = regime_df[["Date", "CCPI_yoy_pct", "In_Event_Window", "Event_Count"]].dropna(subset=["CCPI_yoy_pct"]).copy()
regime_df["Regime"] = regime_df["In_Event_Window"].map({True: "Event Window", False: "Non-Event"})

fig = px.box(
    regime_df,
    x="Regime",
    y="CCPI_yoy_pct",
    points="all",
    color="Regime",
    title="CCPI YoY Inflation: Event Windows vs Non-Event Periods",
)
fig.update_layout(template="plotly_white", yaxis_title="YoY %", showlegend=False, dragmode="pan")
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

fig2 = px.scatter(
    regime_df,
    x="Event_Count",
    y="CCPI_yoy_pct",
    color="Regime",
    title="CCPI YoY Inflation vs Number of Overlapping Events",
    labels={"Event_Count": "Number of Active Events", "CCPI_yoy_pct": "YoY %"},
)
fig2.update_layout(template="plotly_white", dragmode="pan")
fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

### Statistical Testing


In [33]:
# Coverage of time since 2000 by event windows vs non-event periods
coverage_df = monthly.loc[monthly["Date"] >= "2000-01-01", ["Date", "In_Event_Window"]].copy()

total_months = len(coverage_df)
event_months = int(coverage_df["In_Event_Window"].sum())
non_event_months = total_months - event_months

coverage_summary = pd.DataFrame(
    {
        "Period": ["Covered by >=1 event", "Not covered by events", "Total"],
        "Months": [event_months, non_event_months, total_months],
    }
)

coverage_summary["Years"] = coverage_summary["Months"] / 12
coverage_summary["Share_pct"] = (coverage_summary["Months"] / total_months * 100).round(2)

print(f"Date range: {coverage_df['Date'].min().date()} to {coverage_df['Date'].max().date()}")
coverage_summary

Date range: 2000-01-01 to 2025-12-01


,Period,Months,Years,Share_pct
0,Covered by >=1 event,234,19.5,75.0
1,Not covered by events,78,6.5,25.0
2,Total,312,26.0,100.0


### Structural Break, Event Impact, and Granger Causality

In [9]:
import numpy as np
import statsmodels.api as sm

inflation_cols = [
    col for col in ["CCPI_yoy_pct", "CPI_A_yoy_pct", "CPI_B_yoy_pct", "CPI_C_yoy_pct"]
    if col in monthly.columns
]

if not inflation_cols:
    raise ValueError("No CPI YoY series found for statistical testing.")

event_windows = {
    "sars": ("2003-03-01", "2003-07-01"),
    "gfc": ("2008-09-01", "2009-12-01"),
    "covid": ("2020-01-01", "2021-03-01"),
    "tradewar": ("2018-03-01", "2019-12-01"),
}

def build_test_df(series_col):
    test_df = monthly[["Date", series_col]].dropna().copy().sort_values("Date")
    test_df = test_df.rename(columns={series_col: "inflation"})

    for event_name, (start, end) in event_windows.items():
        test_df[event_name] = (
            (test_df["Date"] >= pd.to_datetime(start))
            & (test_df["Date"] <= pd.to_datetime(end))
        ).astype(int)

    test_df["t"] = np.arange(len(test_df))
    test_df["series"] = series_col
    return test_df

all_test_df = {col: build_test_df(col) for col in inflation_cols}
print("Testing series:", inflation_cols)
for col, df_i in all_test_df.items():
    print(f"{col}: {df_i['Date'].min().date()} to {df_i['Date'].max().date()} ({len(df_i)} obs)")


Testing series: ['CCPI_yoy_pct', 'CPI_A_yoy_pct', 'CPI_B_yoy_pct', 'CPI_C_yoy_pct']
CCPI_yoy_pct: 2000-01-01 to 2025-12-01 (312 obs)
CPI_A_yoy_pct: 2000-01-01 to 2025-12-01 (312 obs)
CPI_B_yoy_pct: 2000-01-01 to 2025-12-01 (312 obs)
CPI_C_yoy_pct: 2000-01-01 to 2025-12-01 (312 obs)


In [11]:
def chow_test_at_date(df, break_date):
    break_date = pd.to_datetime(break_date)
    split_idx = int((df["Date"] < break_date).sum())

    if split_idx < 24 or (len(df) - split_idx) < 24:
        return {"Break_Date": break_date.date(), "Status": "Insufficient observations"}

    y = df["inflation"]
    X = sm.add_constant(df[["t"]])
    k = X.shape[1]

    pooled = sm.OLS(y, X).fit()
    first = sm.OLS(y.iloc[:split_idx], X.iloc[:split_idx]).fit()
    second = sm.OLS(y.iloc[split_idx:], X.iloc[split_idx:]).fit()

    rss_pooled = float(np.sum(pooled.resid ** 2))
    rss_split = float(np.sum(first.resid ** 2) + np.sum(second.resid ** 2))
    n = len(df)

    denominator_df = n - 2 * k
    if denominator_df <= 0 or rss_split <= 0:
        return {"Break_Date": break_date.date(), "Status": "Invalid degrees of freedom"}

    f_stat = ((rss_pooled - rss_split) / k) / (rss_split / denominator_df)
    p_val = 1 - f.cdf(f_stat, k, denominator_df)

    return {
        "Break_Date": break_date.date(),
        "F_stat": f_stat,
        "p_value": p_val,
        "Split_Index": split_idx,
        "Status": "OK",
    }

structural_rows = []
chow_rows = []
pelt_rows = []
chow_candidates = ["2003-03-01", "2008-09-01", "2020-01-01"]

for series_col, test_df in all_test_df.items():
    ols_trend = sm.OLS(test_df["inflation"], sm.add_constant(test_df[["t"]])).fit()
    cusum_stat, cusum_pvalue, cusum_crit = breaks_cusumolsresid(
        ols_trend.resid,
        ddof=int(ols_trend.df_model) + 1,
    )
    crit_5pct = np.nan
    if len(cusum_crit) >= 2 and isinstance(cusum_crit[1], tuple) and len(cusum_crit[1]) >= 2:
        crit_5pct = float(cusum_crit[1][1])

    structural_rows.append(
        {
            "Series": series_col,
            "CUSUM_stat": float(cusum_stat),
            "CUSUM_p_value": float(cusum_pvalue),
            "CUSUM_crit_5pct": crit_5pct,
        }
    )

    for d in chow_candidates:
        result = chow_test_at_date(test_df, d)
        result["Series"] = series_col
        chow_rows.append(result)

    try:
        signal = test_df["inflation"].dropna().values
        model = rpt.Pelt(model="rbf").fit(signal)
        break_idx = model.predict(pen=5)
        for idx in break_idx:
            break_date = test_df.iloc[min(max(idx - 1, 0), len(test_df) - 1)]["Date"].date()
            pelt_rows.append({"Series": series_col, "Break_Index": idx, "Break_Date": break_date})
    except ImportError:
        pelt_rows.append({"Series": series_col, "Break_Index": None, "Break_Date": "ruptures_not_installed"})

structural_summary = pd.DataFrame(structural_rows)
chow_results = pd.DataFrame(chow_rows)
pelt_breaks = pd.DataFrame(pelt_rows)

display(structural_summary.sort_values("CUSUM_p_value"))
display(chow_results.sort_values(["Series", "Break_Date"]))
display(pelt_breaks.sort_values(["Series", "Break_Index"], na_position="last"))

,Series,CUSUM_stat,CUSUM_p_value,CUSUM_crit_5pct
3,CPI_C_yoy_pct,3.862752,2.192509e-13,1.36
2,CPI_B_yoy_pct,3.749399,1.231439e-12,1.36
0,CCPI_yoy_pct,3.590531,1.268408e-11,1.36
1,CPI_A_yoy_pct,2.756188,5.043483e-07,1.36


,Break_Date,F_stat,p_value,Split_Index,Status,Series
0,2003-03-01,65.933933,1.110223e-16,38,OK,CCPI_yoy_pct
1,2008-09-01,223.766409,1.110223e-16,104,OK,CCPI_yoy_pct
2,2020-01-01,69.105376,1.110223e-16,240,OK,CCPI_yoy_pct
3,2003-03-01,35.340084,1.521006e-14,38,OK,CPI_A_yoy_pct
4,2008-09-01,83.179609,1.110223e-16,104,OK,CPI_A_yoy_pct
5,2020-01-01,48.921468,1.110223e-16,240,OK,CPI_A_yoy_pct
6,2003-03-01,74.337856,1.110223e-16,38,OK,CPI_B_yoy_pct
7,2008-09-01,278.191813,1.110223e-16,104,OK,CPI_B_yoy_pct
8,2020-01-01,70.654073,1.110223e-16,240,OK,CPI_B_yoy_pct
9,2003-03-01,75.152391,1.110223e-16,38,OK,CPI_C_yoy_pct


,Series,Break_Index,Break_Date
0,CCPI_yoy_pct,55,2004-07-01
1,CCPI_yoy_pct,130,2010-10-01
2,CCPI_yoy_pct,185,2015-05-01
3,CCPI_yoy_pct,312,2025-12-01
4,CPI_A_yoy_pct,55,2004-07-01
5,CPI_A_yoy_pct,130,2010-10-01
6,CPI_A_yoy_pct,185,2015-05-01
7,CPI_A_yoy_pct,312,2025-12-01
8,CPI_B_yoy_pct,5,2000-05-01
9,CPI_B_yoy_pct,50,2004-02-01


In [12]:
dummy_cols = ["covid", "gfc", "tradewar", "sars"]
event_model_rows = []

for series_col, test_df in all_test_df.items():
    X = sm.add_constant(test_df[dummy_cols])
    y = test_df["inflation"]
    event_model = sm.OLS(y, X, missing="drop").fit(cov_type="HC3")

    tmp = pd.DataFrame(
        {
            "Series": series_col,
            "variable": event_model.params.index,
            "coef": event_model.params.values,
            "std_err": event_model.bse.values,
            "t": event_model.tvalues.values,
            "p_value": event_model.pvalues.values,
            "r2": event_model.rsquared,
            "adj_r2": event_model.rsquared_adj,
        }
    )
    event_model_rows.append(tmp)

event_coef_table = pd.concat(event_model_rows, ignore_index=True)
display(event_coef_table.sort_values(["Series", "variable"]))

,Series,variable,coef,std_err,t,p_value,r2,adj_r2
0,CCPI_yoy_pct,const,1.624803,0.159705,10.173777,2.596413e-24,0.076472,0.064439
1,CCPI_yoy_pct,covid,-1.151470,0.446005,-2.581741,9.830342e-03,0.076472,0.064439
2,CCPI_yoy_pct,gfc,-0.612303,0.423196,-1.446856,1.479373e-01,0.076472,0.064439
4,CCPI_yoy_pct,sars,-4.324803,0.465570,-9.289259,1.553656e-20,0.076472,0.064439
3,CCPI_yoy_pct,tradewar,1.047924,0.186820,5.609273,2.031779e-08,0.076472,0.064439
5,CPI_A_yoy_pct,const,1.920079,0.183338,10.472879,1.150881e-25,0.072741,0.060660
6,CPI_A_yoy_pct,covid,-1.693412,0.918510,-1.843651,6.523397e-02,0.072741,0.060660
7,CPI_A_yoy_pct,gfc,-1.557579,0.533252,-2.920907,3.490135e-03,0.072741,0.060660
9,CPI_A_yoy_pct,sars,-4.080079,0.491796,-8.296290,1.074132e-16,0.072741,0.060660
8,CPI_A_yoy_pct,tradewar,1.129921,0.218417,5.173236,2.300739e-07,0.072741,0.060660


In [13]:
def event_vs_rest_ttest(df, dummy_col):
    event_vals = df.loc[df[dummy_col] == 1, "inflation"].dropna()
    rest_vals = df.loc[df[dummy_col] == 0, "inflation"].dropna()

    t_stat, p_val = ttest_ind(event_vals, rest_vals, equal_var=False, nan_policy="omit")
    return {
        "Event": dummy_col,
        "Event_n": len(event_vals),
        "Rest_n": len(rest_vals),
        "Event_mean": event_vals.mean(),
        "Rest_mean": rest_vals.mean(),
        "Mean_Diff": event_vals.mean() - rest_vals.mean(),
        "t_stat": t_stat,
        "p_value": p_val,
    }

ttest_rows = []
for series_col, test_df in all_test_df.items():
    for col in dummy_cols:
        row = event_vs_rest_ttest(test_df, col)
        row["Series"] = series_col
        ttest_rows.append(row)

ttest_results = pd.DataFrame(ttest_rows)
ttest_results.sort_values(["Series", "p_value"])

,Event,Event_n,Rest_n,Event_mean,Rest_mean,Mean_Diff,t_stat,p_value,Series
2,tradewar,22,290,2.672727,1.456897,1.215831,6.947016,7.369138e-11,CCPI_yoy_pct
3,sars,5,307,-2.700000,1.611726,-4.311726,-10.406038,1.354492e-04,CCPI_yoy_pct
0,covid,15,297,0.473333,1.596633,-1.123300,-2.630859,1.709595e-02,CCPI_yoy_pct
1,gfc,16,296,1.012500,1.571284,-0.558784,-1.376784,1.840953e-01,CCPI_yoy_pct
6,tradewar,22,290,3.050000,1.676207,1.373793,6.587218,6.045338e-10,CPI_A_yoy_pct
7,sars,5,307,-2.160000,1.837134,-3.997134,-9.098337,1.841646e-04,CPI_A_yoy_pct
5,gfc,16,296,0.362500,1.849324,-1.486824,-2.897074,9.301623e-03,CPI_A_yoy_pct
4,covid,15,297,0.226667,1.851178,-1.624512,-1.835917,8.625298e-02,CPI_A_yoy_pct
10,tradewar,22,290,2.531818,1.397586,1.134232,6.599562,4.650799e-10,CPI_B_yoy_pct
11,sars,5,307,-2.860000,1.548208,-4.408208,-11.208415,8.432392e-05,CPI_B_yoy_pct


Results

Structural break evidence is strong: CUSUM statistic 3.5905 with p-value 0.0.

Chow tests at 2003-03, 2008-09, and 2020-01 are all highly significant (all p-values about 1.11e-16), supporting your event-date choices.

Automatic breakpoint detection (PELT, rbf, pen=5) detected breaks near 2004-07, 2010-10, 2015-05, and sample end (2025-12).

Event-dummy OLS (HC3 robust SE): COVID coefficient about -1.151 (p=0.0098), Trade War about +1.048 (p=2.03e-08), SARS about -4.325 (p=1.55e-20), GFC not significant at 5% (p=0.148).

Welch t-tests (event vs rest): Trade War, SARS, and COVID are significant; GFC is not significant at 5%.

### ADF and Granger Causality (with auto-differencing)

In [15]:
import warnings

series_for_granger = [col for col in inflation_cols if col in monthly.columns]
granger_df = monthly[["Date"] + series_for_granger].dropna().sort_values("Date").reset_index(drop=True)

def adf_pvalue(series):
    series = pd.Series(series).dropna()
    if len(series) < 20:
        return np.nan
    return adfuller(series, autolag="AIC")[1]

def make_stationary(series, alpha=0.05, max_diff=3):
    cur = pd.Series(series).dropna()
    diff_order = 0
    p_val = adf_pvalue(cur)

    while pd.notna(p_val) and p_val > alpha and diff_order < max_diff:
        cur = cur.diff().dropna()
        diff_order += 1
        p_val = adf_pvalue(cur)

    return cur, diff_order, p_val

# ADF + auto-differencing per CPI series
stationary_results = []
stationary_series = {}
diff_orders = {}

for col in series_for_granger:
    s, d, p = make_stationary(granger_df[col], alpha=0.05, max_diff=3)
    stationary_series[col] = s
    diff_orders[col] = d
    stationary_results.append(
        {
            "Series": col,
            "Differencing_Order": d,
            "ADF_p_value_after_diff": p,
            "Stationary_at_5pct": bool(pd.notna(p) and p <= 0.05),
        }
    )

adf_results = pd.DataFrame(stationary_results)
display(adf_results)

# Align stationary series for Granger testing
stationary_panel = pd.concat(stationary_series, axis=1).dropna()

# Pairwise Granger causality among CPI series
max_lag = 6
granger_rows = []

for target in series_for_granger:
    for driver in series_for_granger:
        if target == driver:
            continue
        pair_df = stationary_panel[[target, driver]].dropna()
        if len(pair_df) < (max_lag + 15):
            granger_rows.append(
                {
                    "From": driver,
                    "To": target,
                    "Best_Lag": np.nan,
                    "Min_p_value": np.nan,
                    "Significant_5pct": False,
                    "Obs": len(pair_df),
                }
            )
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            test_out = grangercausalitytests(pair_df, maxlag=max_lag, verbose=False)
        pvals = {lag: test_out[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1)}
        best_lag = min(pvals, key=pvals.get)
        min_p = pvals[best_lag]
        granger_rows.append(
            {
                "From": driver,
                "To": target,
                "Best_Lag": best_lag,
                "Min_p_value": float(min_p),
                "Significant_5pct": bool(min_p <= 0.05),
                "Obs": len(pair_df),
            }
        )

granger_results = pd.DataFrame(granger_rows).sort_values(["To", "Min_p_value"], na_position="last")
display(granger_results)

# Interactive heatmap of Granger min p-values (rows=From, cols=To)
pval_matrix = granger_results.pivot(index="From", columns="To", values="Min_p_value")
fig = px.imshow(
    pval_matrix,
    color_continuous_scale="RdYlGn_r",
    zmin=0,
    zmax=0.10,
    title="Pairwise Granger Causality (Min p-value across lags)",
    labels={"color": "Min p-value"},
)
fig.update_layout(template="plotly_white", dragmode="pan")
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

# Interactive bar chart of differencing orders used for stationarity
fig2 = px.bar(
    adf_results.sort_values("Differencing_Order", ascending=False),
    x="Series",
    y="Differencing_Order",
    color="Stationary_at_5pct",
    title="ADF-based Differencing Order by CPI Series",
    text="Differencing_Order",
)
fig2.update_layout(template="plotly_white", dragmode="pan")
fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

,Series,Differencing_Order,ADF_p_value_after_diff,Stationary_at_5pct
0,CCPI_yoy_pct,1,1.433033e-08,True
1,CPI_A_yoy_pct,1,1.345770e-13,True
2,CPI_B_yoy_pct,1,7.104719e-11,True
3,CPI_C_yoy_pct,1,2.148908e-06,True


,From,To,Best_Lag,Min_p_value,Significant_5pct,Obs
0,CPI_A_yoy_pct,CCPI_yoy_pct,6,0.000023,True,311
1,CPI_B_yoy_pct,CCPI_yoy_pct,6,0.000052,True,311
2,CPI_C_yoy_pct,CCPI_yoy_pct,6,0.000547,True,311
4,CPI_B_yoy_pct,CPI_A_yoy_pct,6,0.000460,True,311
3,CCPI_yoy_pct,CPI_A_yoy_pct,6,0.001042,True,311
5,CPI_C_yoy_pct,CPI_A_yoy_pct,6,0.008699,True,311
7,CPI_A_yoy_pct,CPI_B_yoy_pct,6,0.125349,False,311
8,CPI_C_yoy_pct,CPI_B_yoy_pct,3,0.162110,False,311
6,CCPI_yoy_pct,CPI_B_yoy_pct,6,0.220106,False,311
10,CPI_A_yoy_pct,CPI_C_yoy_pct,6,0.012245,True,311
